In [0]:
%python
dbutils.widgets.text("catalog","fk_1368")
catalog=dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.stores USING delta COMMENT 'raw stores csv files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.stores
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at
    FROM '/Volumes/{catalog}/landing/raw/stores/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.customers USING delta COMMENT 'raw customers csv files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.customers
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at
    FROM '/Volumes/{catalog}/landing/raw/customers/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.products USING delta COMMENT 'raw products json files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.products
  FROM '/Volumes/{catalog}/landing/raw/products/'
  FILEFORMAT = JSON
  FILES = ('products_2026-03-01.json')
  FORMAT_OPTIONS ('multiLine' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# Create the target Bronze table
source_path = f"/Volumes/{catalog}/landing/raw/orders/"
target_table = f"{catalog}.bronze.orders"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/checkpoints/orders"
schema_path = f"/Volumes/{catalog}/ops/pipeline_state/schema/orders"

In [0]:
# Read files 
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# Add metadata columns for file path and ingestion timestamp
from pyspark.sql.functions import col, current_timestamp

df_stream = (
    df_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# Write to Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
SELECT COUNT(*) AS total_rows

FROM IDENTIFIER(:catalog || '.bronze.orders');